In [ ]:
import numpy as np
import evaluate
from transformers import BitsAndBytesConfig
from transformers import AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer, pipeline
from datasets import load_dataset

In [ ]:
ds = load_dataset("mteb/tweet_sentiment_extraction")
ds

In [ ]:
assert len(ds["train"].unique("id")) == len(ds["train"])
assert len(ds["test"].unique("id")) == len(ds["test"])

In [ ]:
ds = ds.filter(lambda x: x["text"] is not None and x["label"] is not None)
ds

In [ ]:
ds_final = ds["train"].train_test_split(test_size = 0.2, seed = 42)
ds_final["validation"] = ds_final.pop("test")
ds_final["test"] = ds["test"]
ds_final

In [ ]:
checkpoint_id = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint_id)

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)
tokenized_datasets = ds_final.map(tokenize_function, batched=True)
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")

In [ ]:
tokenized_datasets

In [ ]:
print(tokenized_datasets["train"][0].keys())

In [ ]:
tokenized_datasets = tokenized_datasets.remove_columns(['text', 'label_text', 'id'])

In [ ]:
tokenized_datasets.save_to_disk("tokenized_datasets")